New Notebook Created by Jupyter MCP Server

In [1]:
import pandas as pd

df = pd.read_csv('MSE/Term1/ml/tasks/3/deliverytime.csv')
# drop ID columns
cols_to_drop = ['ID','Delivery_person_ID']
df = df.drop(columns=cols_to_drop)
print('Remaining feature count (including target):', df.shape[1])
print('Remaining feature count (excluding target Time_taken_min):', df.shape[1]-1)
print(df.columns.tolist())

FileNotFoundError: [Errno 2] No such file or directory: 'MSE/Term1/ml/tasks/3/deliverytime.csv'

In [2]:
import os, glob
print(os.getcwd())
print(glob.glob('*'))

/home/iliayar/Repos/ITMO/MSE/Term1/ml/tasks
['3.3.1', '3.5.2', 'flake.nix', '3.4.1', '3.5.3', '3.5.4', 'task3.ipynb', '3.3.2', '3.1.1', '3.3.3', 'flake.lock', '3.2.1', '3', '3.2.2']


In [3]:
df = pd.read_csv('3/deliverytime.csv')
cols_to_drop = ['ID','Delivery_person_ID']
df = df.drop(columns=cols_to_drop)
print('Remaining feature count (including target):', df.shape[1])
print('Remaining feature count (excluding target Time_taken_min):', df.shape[1]-1)
print(df.columns.tolist())

Remaining feature count (including target): 10
Remaining feature count (excluding target Time_taken_min): 9
['Delivery_person_Age', 'Delivery_person_Ratings', 'distance_meters', 'order_Drinks', 'order_Meal', 'order_Snack', 'vehicle_electric_scooter', 'vehicle_motorcycle', 'vehicle_scooter', 'Time_taken_min']


In [4]:
# Filter out orders with distance > 100km (100000 meters)
df_filtered = df[df['distance_meters'] <= 100000]
print('Filtered dataset size (rows):', df_filtered.shape[0])
print('Columns:', df_filtered.shape[1])

Filtered dataset size (rows): 45162
Columns: 10


In [5]:
max_distance = df_filtered['distance_meters'].max()
print('Max distance_meters after filtering:', max_distance)

Max distance_meters after filtering: 20969.489380087085


In [6]:
from sklearn.model_selection import train_test_split

features = ['Delivery_person_Age','Delivery_person_Ratings','distance_meters']
X = df_filtered[features]
y = df_filtered['Time_taken_min']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=25)
print('Mean age (train):', X_train['Delivery_person_Age'].mean())

Mean age (train): 29.54385673558637


In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print('MAE:', mae)

MAE: 6.3319638433910095


In [8]:
from sklearn.preprocessing import MinMaxScaler

# Full feature set (excluding IDs)
X_full = df_filtered.drop(columns=['Time_taken_min'])
y_full = df_filtered['Time_taken_min']
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(X_full, y_full, test_size=0.2, random_state=25)

scaler = MinMaxScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_full), columns=X_train_full.columns, index=X_train_full.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_full), columns=X_test_full.columns, index=X_test_full.index)
print('Scaled mean age (train):', X_train_scaled['Delivery_person_Age'].mean())

Scaled mean age (train): 0.4155387638738963


In [9]:
lr_full = LinearRegression()
lr_full.fit(X_train_scaled, y_train_full)
y_pred_full = lr_full.predict(X_test_scaled)
mae_full = mean_absolute_error(y_test_full, y_pred_full)
print('MAE with all features scaled:', mae_full)

MAE with all features scaled: 6.258556157580087


In [10]:
# Create classification target
df_class = df_filtered.copy()
df_class['is_long_Delivery'] = (df_class['Time_taken_min'] > 30).astype(int)
fast_ratio = (df_class['is_long_Delivery'] == 0).mean()
print('Fast deliveries ratio:', fast_ratio)
print('Count fast:', (df_class['is_long_Delivery']==0).sum(), 'Total:', len(df_class))

Fast deliveries ratio: 0.7012975510384837
Count fast: 31672 Total: 45162


In [11]:
# Remove Time_taken_min for classification
clf_df = df_class.drop(columns=['Time_taken_min'])
print('Columns for classification:', clf_df.columns.tolist())
print('Shape:', clf_df.shape)

Columns for classification: ['Delivery_person_Age', 'Delivery_person_Ratings', 'distance_meters', 'order_Drinks', 'order_Meal', 'order_Snack', 'vehicle_electric_scooter', 'vehicle_motorcycle', 'vehicle_scooter', 'is_long_Delivery']
Shape: (45162, 10)


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

X_clf = clf_df.drop(columns=['is_long_Delivery'])
y_clf = clf_df['is_long_Delivery']
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(X_clf, y_clf, test_size=0.2, random_state=25, stratify=y_clf)

knn = KNeighborsClassifier()
knn.fit(X_train_clf, y_train_clf)
y_pred_clf = knn.predict(X_test_clf)
acc = accuracy_score(y_test_clf, y_pred_clf)
print('Accuracy:', acc)

Accuracy: 0.7301007417247869
